In [9]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [1]:
words = open('names.txt','r').read().splitlines()


In [2]:
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [3]:
len(words)

32033

In [4]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [14]:
#build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one? Padding with '.' which is mapped to 0
X, Y = [], []
for w in words[:5]: # take the first 1000 words to train on
    print(w)
    context = [0] * block_size # initial context is all '.'
    for ch in w + '.': # for each character in the word + the stop character
        ix = stoi[ch] # get the index of the character
        X.append(context) # add the current context to X
        Y.append(ix) # add the index of the next character to Y
        print(''.join(itos[i] for i in context), '->', itos[ix]) # print the current context and the next character
        context = context[1:] + [ix] # slide the context window and add the new character
        
X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... -> e
..e -> m
.em -> m
emm -> a
mma -> .
olivia
... -> o
..o -> l
.ol -> i
oli -> v
liv -> i
ivi -> a
via -> .
ava
... -> a
..a -> v
.av -> a
ava -> .
isabella
... -> i
..i -> s
.is -> a
isa -> b
sab -> e
abe -> l
bel -> l
ell -> a
lla -> .
sophia
... -> s
..s -> o
.so -> p
sop -> h
oph -> i
phi -> a
hia -> .


In [15]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [16]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1]])

In [17]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [21]:
#We will cram the 27 possible chartacters into a 2d space, and then learn a linear classifier on top of that
C = torch.rand(27,2) # lookup table with 27 rows anbd 2 columns, initialized randomly


In [23]:
C[5]

tensor([0.0072, 0.4017])

In [25]:
#F.one_hot(5, num_classes=27) This will not work because it returns a 1d tensor, and we need a 2d tensor to do the matrix multiplication with C. We can use unsqueeze to add an extra dimension to the tensor.
F.one_hot(torch.tensor(5), num_classes=27)

tensor([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0])

In [27]:
F.one_hot(Y, num_classes=27).dtype

torch.int64

In [29]:
#If we take the one hot and multiply it by C, we get an error because one hot is a long integer while C is a float, Python does not know how to multiply a long integer by a float, we need to convert the one hot to a float tensor.
F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([0.0072, 0.4017])

#since they are the same result from the indexing and usin one hotm we will stick with the indexing method because it is more efficient and easier to read. We can use the one hot method to check our work and make sure we are getting the same result.

#How do we embbed all the 32 ,3 integers stores in array X, we can index with a list or tensor oif integers

In [32]:
C[[5,6,7]]

tensor([[0.0072, 0.4017],
        [0.5338, 0.4488],
        [0.3417, 0.3093]])

In [35]:
C[torch.tensor([[5,6,7]])]

tensor([[[0.0072, 0.4017],
         [0.5338, 0.4488],
         [0.3417, 0.3093]]])

In [36]:
C[torch.tensor([[5,6,7,7,7,7,7,7]])]

tensor([[[0.0072, 0.4017],
         [0.5338, 0.4488],
         [0.3417, 0.3093],
         [0.3417, 0.3093],
         [0.3417, 0.3093],
         [0.3417, 0.3093],
         [0.3417, 0.3093],
         [0.3417, 0.3093]]])

In [40]:
# We can aklso index with a 2d  tensor as following:
C[X].shape

torch.Size([32, 3, 2])

In [41]:
X[13,2]

tensor(1)

In [42]:
C[X][13,2]

tensor([0.3313, 0.4200])

In [43]:
C[1]

tensor([0.3313, 0.4200])

In [44]:
#So, to embed simultaneously all the integers in X,we can simply do C[X], which will give us a 3d tensor of shape (number of examples, block_size, embedding_size) in our case (number of examples, 3, 2)
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [45]:
w1 = torch.randn((6,100)) # first layer weights
b1 = torch.randn(100) # first layer bias

In [47]:
# Typically we take the input multiply with the weight and add bias, the issue us the embeddings are stacked up in the dimensions of the input tensor.
# How do we transform the 32,3,2 into 32,6 to be abke to calculate the embedding 
# We need to concatenate the inout to be able to do emb @ w1 + b
# We will want to retrieve all the 3 parts in the networks and concetenate them
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], dim=1) # this will give us the three parts of the embedding, we can then concatenate them together to get a 32,6 tensor that we can multiply with w1 and add b1. We can use torch.cat to concatenate the tensors together. We need to specify the dimension to concatenate on, in this case we want to concatenate on the last dimension, which is the embedding dimension. So we will concatenate on dimension 2. We also need to specify the order of the tensors we want to concatenate, in this case we want to concatenate the first part of the embedding, then the second part, then the third part. So we will concatenate emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]].




tensor([[0.2629, 0.1757, 0.2629, 0.1757, 0.2629, 0.1757],
        [0.2629, 0.1757, 0.2629, 0.1757, 0.0072, 0.4017],
        [0.2629, 0.1757, 0.0072, 0.4017, 0.7261, 0.5546],
        [0.0072, 0.4017, 0.7261, 0.5546, 0.7261, 0.5546],
        [0.7261, 0.5546, 0.7261, 0.5546, 0.3313, 0.4200],
        [0.2629, 0.1757, 0.2629, 0.1757, 0.2629, 0.1757],
        [0.2629, 0.1757, 0.2629, 0.1757, 0.0848, 0.9588],
        [0.2629, 0.1757, 0.0848, 0.9588, 0.3136, 0.1721],
        [0.0848, 0.9588, 0.3136, 0.1721, 0.3675, 0.6000],
        [0.3136, 0.1721, 0.3675, 0.6000, 0.7265, 0.9656],
        [0.3675, 0.6000, 0.7265, 0.9656, 0.3675, 0.6000],
        [0.7265, 0.9656, 0.3675, 0.6000, 0.3313, 0.4200],
        [0.2629, 0.1757, 0.2629, 0.1757, 0.2629, 0.1757],
        [0.2629, 0.1757, 0.2629, 0.1757, 0.3313, 0.4200],
        [0.2629, 0.1757, 0.3313, 0.4200, 0.7265, 0.9656],
        [0.3313, 0.4200, 0.7265, 0.9656, 0.3313, 0.4200],
        [0.2629, 0.1757, 0.2629, 0.1757, 0.2629, 0.1757],
        [0.262

In [49]:
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], dim=1).shape
#Since we do not want to keeop chaning when we modify the context/block soize, we will need to use unbind to remove a dimension

torch.Size([32, 6])

In [50]:
torch.unbind(emb, dim=1) # this will give us a list of 3 tensors, each of shape (number of examples, embedding_size), we can then concatenate them together to get a tensor of shape (number of examples, block_size * embedding_size) that we can multiply with w1 and add b1. We can use torch.cat to concatenate the tensors together. We need to specify the dimension to concatenate on, in this case we want to concatenate on the last dimension, which is the embedding dimension. So we will concatenate on dimension 1. We also need to specify the order of the tensors we want to concatenate, in this case we want to concatenate the first part of the embedding, then the second part, then the third part. So we will concatenate emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]].

(tensor([[0.2629, 0.1757],
         [0.2629, 0.1757],
         [0.2629, 0.1757],
         [0.0072, 0.4017],
         [0.7261, 0.5546],
         [0.2629, 0.1757],
         [0.2629, 0.1757],
         [0.2629, 0.1757],
         [0.0848, 0.9588],
         [0.3136, 0.1721],
         [0.3675, 0.6000],
         [0.7265, 0.9656],
         [0.2629, 0.1757],
         [0.2629, 0.1757],
         [0.2629, 0.1757],
         [0.3313, 0.4200],
         [0.2629, 0.1757],
         [0.2629, 0.1757],
         [0.2629, 0.1757],
         [0.3675, 0.6000],
         [0.4904, 0.5236],
         [0.3313, 0.4200],
         [0.1310, 0.9687],
         [0.0072, 0.4017],
         [0.3136, 0.1721],
         [0.2629, 0.1757],
         [0.2629, 0.1757],
         [0.2629, 0.1757],
         [0.4904, 0.5236],
         [0.0848, 0.9588],
         [0.2000, 0.7318],
         [0.5004, 0.8970]]),
 tensor([[0.2629, 0.1757],
         [0.2629, 0.1757],
         [0.0072, 0.4017],
         [0.7261, 0.5546],
         [0.7261, 0.5546],

In [71]:
torch.cat(torch.unbind(emb, dim=1), dim=1).shape  #does not matter blok size with this code
# Using torch,cat is inefficient becasuse new memory is created, creates a whole new tensor with new storage, cannot just manipoulate the view attributes.

torch.Size([32, 6])

In [55]:
a = torch.arange(18)
a

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [59]:
a.view(3,3,2)

tensor([[[ 0,  1],
         [ 2,  3],
         [ 4,  5]],

        [[ 6,  7],
         [ 8,  9],
         [10, 11]],

        [[12, 13],
         [14, 15],
         [16, 17]]])

In [ ]:
a.storage() #eah tensotr has a storage which is a 1d array that contains all the data of the tensor, and the tensor itself is just a view on that storage with a certain shape and stride. The storage is what is actually stored in memory, and the tensor is just a way to interpret that data. The view method allows us to change the shape of the tensor without changing the underlying data, while the storage method allows us to access the underlying data directly.

C:\Users\brian\AppData\Local\Temp\ipykernel_8452\214256462.py:1: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  a.storage()


 0
 1
 2
 3
 4
 5
 6
 7
 8
 9
 10
 11
 12
 13
 14
 15
 16
 17
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 18]

In [61]:
emb.view(32,6)

tensor([[0.2629, 0.1757, 0.2629, 0.1757, 0.2629, 0.1757],
        [0.2629, 0.1757, 0.2629, 0.1757, 0.0072, 0.4017],
        [0.2629, 0.1757, 0.0072, 0.4017, 0.7261, 0.5546],
        [0.0072, 0.4017, 0.7261, 0.5546, 0.7261, 0.5546],
        [0.7261, 0.5546, 0.7261, 0.5546, 0.3313, 0.4200],
        [0.2629, 0.1757, 0.2629, 0.1757, 0.2629, 0.1757],
        [0.2629, 0.1757, 0.2629, 0.1757, 0.0848, 0.9588],
        [0.2629, 0.1757, 0.0848, 0.9588, 0.3136, 0.1721],
        [0.0848, 0.9588, 0.3136, 0.1721, 0.3675, 0.6000],
        [0.3136, 0.1721, 0.3675, 0.6000, 0.7265, 0.9656],
        [0.3675, 0.6000, 0.7265, 0.9656, 0.3675, 0.6000],
        [0.7265, 0.9656, 0.3675, 0.6000, 0.3313, 0.4200],
        [0.2629, 0.1757, 0.2629, 0.1757, 0.2629, 0.1757],
        [0.2629, 0.1757, 0.2629, 0.1757, 0.3313, 0.4200],
        [0.2629, 0.1757, 0.3313, 0.4200, 0.7265, 0.9656],
        [0.3313, 0.4200, 0.7265, 0.9656, 0.3313, 0.4200],
        [0.2629, 0.1757, 0.2629, 0.1757, 0.2629, 0.1757],
        [0.262

In [63]:
emb.view(32,6) == torch.cat(torch.unbind(emb, dim=1), dim=1) # These 2 are equal

tensor([[True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, T

In [70]:
h = emb.view(emb.shape[0],6) @ w1 + b1 # this is the same as torch.cat(torch.unbind(emb, dim=1), dim=1) @ w1 + b1
# Can also use h = emb.view(-1, 6) @ w1 + b1, the -1 will automatically infer the correct size for that dimension based on the other dimensions and the total number of elements in the tensor. This is a convenient way to reshape a tensor when we don't want to manually calculate the size of one of the dimensions.

In [66]:
h

tensor([[-1.2108e+00, -7.2726e-01, -1.0815e+00,  ..., -9.6820e-01,
          1.8843e+00, -2.5857e-01],
        [-7.4873e-01, -4.9745e-01, -1.1442e+00,  ..., -1.6828e+00,
          1.4102e+00, -4.3087e-01],
        [-1.2403e+00, -9.0146e-01, -1.2681e+00,  ..., -3.8102e-01,
          1.6344e+00,  8.3750e-02],
        ...,
        [-2.6111e+00, -1.5186e+00,  4.7901e-02,  ..., -6.8055e-01,
          1.3449e+00, -5.4185e-01],
        [-2.6365e+00, -1.5935e+00,  1.7526e-03,  ..., -8.8460e-01,
          2.0365e+00, -6.4731e-01],
        [-2.7225e+00, -1.5760e+00, -2.3430e-01,  ..., -3.9815e-02,
          2.2129e+00, -2.2546e-01]])

In [72]:
# Most efficient way
h = torch.tanh(emb.view(-1, 6) @ w1 + b1) # this is the same as torch.tanh(torch.cat(torch.unbind(emb, dim=1), dim=1) @ w1 + b1) but more efficient because it does not create a new tensor with new storage, it just manipulates the view attributes of the original tensor.

In [73]:
h

tensor([[-0.8369, -0.6214, -0.7938,  ..., -0.7479,  0.9549, -0.2530],
        [-0.6344, -0.4601, -0.8158,  ..., -0.9332,  0.8875, -0.4061],
        [-0.8455, -0.7170, -0.8533,  ..., -0.3636,  0.9267,  0.0836],
        ...,
        [-0.9893, -0.9085,  0.0479,  ..., -0.5919,  0.8728, -0.4944],
        [-0.9898, -0.9207,  0.0018,  ..., -0.7087,  0.9665, -0.5699],
        [-0.9914, -0.9180, -0.2301,  ..., -0.0398,  0.9764, -0.2217]])

In [74]:
h.shape

torch.Size([32, 100])

In [76]:
#Broadcasting check
(emb.view(-1, 6) @ w1).shape

torch.Size([32, 100])

In [77]:
b1.shape

torch.Size([100])

In [80]:
#32,100
# 1,100, broadcasting will align on the right and take a fake dimension on the left
#Broadcasting will copy vertically all the rows of 32 and do an elementwise addition, sanme bias vector will be added to all the rows, whihc is correct.

In [81]:
h.shape

torch.Size([32, 100])

In [83]:
w2 = torch.randn((100,27)) # second layer weights
b2 = torch.randn((27)) # second layer biases

In [84]:
logits = h @ w2 + b2 # this will give us the logits for each of the 27 possible characters, we can then use these logits to calculate the loss and do backpropagation to update the weights and biases of the network. The shape of logits will be (number of examples, number of classes), in this case (number of examples, 27) because we have 27 possible characters to predict. We can then use these logits to calculate the loss and do backpropagation to update the weights and biases of the network.

In [85]:
logits.shape

torch.Size([32, 27])

In [86]:
counts = logits.exp() # this will give us the counts for each of the 27 possible characters, we can then use these counts to calculate the probabilities and do backpropagation to update the weights and biases of the network. The shape of counts will be (number of examples, number of classes), in this case (number of examples, 27) because we have 27 possible characters to predict. We can then use these counts to calculate the probabilities and do backpropagation to update the weights and biases of the network.

In [87]:
prob = counts/counts.sum(1, keepdim=True) # this will give us the probabilities for each of the 27 possible characters, we can then use these probabilities to calculate the loss and do backpropagation to update the weights and biases of the network. The shape of prob will be (number of examples, number of classes), in this case (number of examples, 27) because we have 27 possible characters to predict. We can then use these probabilities to calculate the loss and do backpropagation to update the weights and biases of the network. We need to sum the counts along the second dimension (the classes) to get the total count for each example, and then we need to keep the dimensions of the result so that we can divide the counts by the total count for each example to get the probabilities. This is done by setting keepdim=True in the sum function.

In [88]:
prob.shape

torch.Size([32, 27])

In [91]:
prob[0].sum() # the probabilities for each example should sum to 1, we can check this by summing the probabilities for the first example and see if it equals 1. This is a good way to check if our probabilities are correct, if they do not sum to 1 then there is likely a bug in our code that we need to fix.

tensor(1.)

In [92]:
Y 

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [93]:
torch.arange(32)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])

In [94]:
prob[torch.arange(32), Y] # this will give us the probabilities for the correct characters for each example, we can then use these probabilities to calculate the loss and do backpropagation to update the weights and biases of the network. The shape of this tensor will be (number of examples,) because we are selecting one probability for each example, which corresponds to the correct character that we are trying to predict. We can then use these probabilities to calculate the loss and do backpropagation to update the weights and biases of the network.    

tensor([4.0435e-05, 1.0521e-10, 1.4447e-11, 3.1475e-09, 7.3360e-12, 5.7801e-06,
        8.3392e-07, 1.5910e-02, 3.8557e-08, 7.8398e-06, 1.1008e-09, 9.3819e-13,
        1.2880e-10, 1.5292e-09, 7.6561e-12, 2.5078e-12, 3.8281e-04, 1.0521e-01,
        8.8149e-10, 1.3965e-09, 1.6665e-04, 1.6091e-10, 8.5444e-05, 5.5520e-11,
        1.1904e-12, 1.3784e-03, 8.2306e-04, 1.2680e-11, 3.2539e-09, 1.9934e-05,
        1.9458e-10, 2.9613e-13])

In [96]:
loss = -prob[torch.arange(32), Y].log().mean() # this will give us the average negative log likelihood loss for the correct characters, we can then use this loss to do backpropagation to update the weights and biases of the network. The shape of this tensor will be a single scalar value because we are taking the mean of the negative log probabilities for all the examples. We can then use this loss to do backpropagation to update the weights and biases of the network.
loss

tensor(17.7936)

#Cleaning up the Code

In [97]:
X.shape,Y.shape #dataset

(torch.Size([32, 3]), torch.Size([32]))

In [100]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [101]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [108]:
emb = C[X] # (32, 3, 10)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 200)
logits = h @ W2 + b2 # (32, 27)
counts = logits.exp() # this will give us the counts for each of the 27 possible characters, we can then use these counts to calculate the probabilities and do backpropagation to update the weights and biases of the network. The shape of counts will be (number of examples, number of classes), in this case (number of examples, 27) because we have 27 possible characters to predict. We can then use these counts to calculate the probabilities and do backpropagation to update the weights and biases of the network.
prob = counts/counts.sum(1, keepdims=True) # this will give us the
loss = -prob[torch.arange(32), Y].log().mean() # this will give us the average negative log likelihood loss for the correct characters, we can then use this loss to do backpropagation to update the weights and biases of the network. The shape of this tensor will be a single scalar value because we are taking the mean of the negative log probabilities for all the examples. We can then use this loss to do backpropagation to update the weights and biases of the network.
loss

tensor(17.7697)